# Experimental Analysis of Matrix Multiplication Algorithms

This notebook connects the experimental pipeline to the algorithmic interpretation. It uses the raw and aggregated CSV files generated by the benchmark targets, validates that the dataset is complete, and then builds visualizations for execution time, tracked heap usage, empirical growth factors and numerical error.

The intended workflow is:

1. `make benchmark-full-resume`
2. `make aggregate-full`
3. run this notebook

The benchmark records `time_seconds` around the multiplication call only. Matrix generation, correctness validation, CSV resume bookkeeping and plotting are outside the timed interval.

## 1. Setup

The notebook uses `pandas`, `numpy`, `matplotlib` and `seaborn`. The Matplotlib cache is redirected to `results/.mplconfig` so the notebook works in sandboxed or remote environments where the default user cache may not be writable.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
ANALYSIS_DIR = RESULTS_DIR / "analysis"
FIGURE_DIR = ANALYSIS_DIR / "notebook_figures"

os.environ.setdefault("MPLCONFIGDIR", str(RESULTS_DIR / ".mplconfig"))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 160
plt.rcParams["axes.titleweight"] = "bold"

RAW_CSV = RESULTS_DIR / "experiment_results_sizes9_pairs30_seed42.csv"
SUMMARY_CSV = RESULTS_DIR / "experiment_summary_sizes9_pairs30_seed42.csv"

RAW_CSV, SUMMARY_CSV

## 2. Load and Validate the Dataset

The raw CSV should contain one row per `(algorithm, matrix_size, pair_id)`. With five algorithms, nine sizes and 30 input pairs, the expected total is `5 * 9 * 30 = 1350` data rows. The aggregated CSV should contain one row per `(algorithm, matrix_size)`, therefore `45` groups.

In [ ]:
raw = pd.read_csv(RAW_CSV)
summary = pd.read_csv(SUMMARY_CSV)

algorithms = [
    "iterative",
    "recursive",
    "hybrid_divide_conquer",
    "strassen",
    "hybrid_strassen",
]
algorithm_labels = {
    "iterative": "Iterative",
    "recursive": "Divide and Conquer",
    "hybrid_divide_conquer": "Hybrid D&C",
    "strassen": "Strassen",
    "hybrid_strassen": "Hybrid Strassen",
}
summary["algorithm_label"] = summary["algorithm"].map(algorithm_labels)
raw["algorithm_label"] = raw["algorithm"].map(algorithm_labels)

expected_rows = len(algorithms) * raw["matrix_size"].nunique() * raw["pair_id"].nunique()
actual_rows = len(raw)
invalid_rows = int((raw["is_correct"] != 1).sum())
completeness = raw.groupby(["algorithm", "matrix_size"]).size().unstack(fill_value=0)

print(f"Raw rows: {actual_rows} / expected {expected_rows}")
print(f"Invalid correctness rows: {invalid_rows}")
display(completeness.loc[algorithms])

assert actual_rows == expected_rows, "Raw CSV is incomplete. Run make benchmark-full-resume and make aggregate-full."
assert invalid_rows == 0, "At least one run failed correctness validation."
assert (completeness == 30).all().all(), "Every algorithm/size group must have 30 samples."

## 3. Overview Table

The table below keeps the most important aggregated fields together: mean execution time, standard deviation, peak tracked heap and maximum observed numerical error. `heap_peak_bytes_mean` is the preferred memory metric because it comes from `tracked_malloc`/`tracked_free`, while process-level RSS fields are more sensitive to allocator and operating-system behavior.

In [ ]:
overview = summary[[
    "algorithm_label",
    "matrix_size",
    "sample_count",
    "time_seconds_mean",
    "time_seconds_stddev",
    "heap_peak_bytes_mean",
    "heap_allocations_mean",
    "max_abs_error_mean",
    "max_rel_error_mean",
]].sort_values(["matrix_size", "algorithm_label"])

display(overview)

## 4. Mean Execution Time

The first comparison uses a log-log plot. For pure asymptotic behavior, doubling `n` should multiply execution time by approximately `8` for `O(n^3)` algorithms and by approximately `7` for Strassen's `O(n^log2(7))` recurrence. Deviations are expected because cache behavior, recursive copying and temporary allocations are major practical effects.

In [ ]:
def save_current_figure(name: str) -> Path:
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    return path

fig, ax = plt.subplots(figsize=(9.5, 5.6))
sns.lineplot(
    data=summary,
    x="matrix_size",
    y="time_seconds_mean",
    hue="algorithm_label",
    marker="o",
    hue_order=[algorithm_labels[a] for a in algorithms],
    ax=ax,
)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Matrix size n")
ax.set_ylabel("Mean time (seconds, log scale)")
ax.set_title("Mean Execution Time by Algorithm")
ax.legend(title="Algorithm", bbox_to_anchor=(1.02, 1), loc="upper left")
save_current_figure("time_seconds_mean_loglog.png")

### Interpretation

The recursive divide-and-conquer implementation follows the expected cubic trend but is consistently much slower than the iterative baseline because it repeatedly allocates, copies and combines submatrices. Pure Strassen reduces the multiplication count and becomes faster than pure divide-and-conquer, but its overhead is still high because it recurses down to `n == 1`.

The hybrid algorithms are the practical winners. The threshold avoids expensive recursive structure on small subproblems, and `hybrid_strassen` becomes the strongest option at the larger tested sizes.

## 5. Raw Sample Distribution

Means are useful, but the raw samples show variability. This plot uses the original per-pair data, not the aggregated CSV.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.8))
sns.boxplot(
    data=raw,
    x="matrix_size",
    y="time_seconds",
    hue="algorithm_label",
    hue_order=[algorithm_labels[a] for a in algorithms],
    ax=ax,
)
ax.set_yscale("log")
ax.set_xlabel("Matrix size n")
ax.set_ylabel("Time (seconds, log scale)")
ax.set_title("Distribution of Execution Time Across Matrix Pairs")
ax.legend(title="Algorithm", bbox_to_anchor=(1.02, 1), loc="upper left")
save_current_figure("time_seconds_distribution.png")

## 6. Empirical Growth Exponents

For consecutive sizes, the empirical exponent is computed as:

```text
log2(T(2n) / T(n))
```

Values near `3` are consistent with cubic growth. Values near `log2(7) ≈ 2.807` are consistent with Strassen-like growth. Small sizes can be noisy, so the larger transitions should receive more weight in the interpretation.

In [ ]:
growth_rows = []
for algorithm, group in summary.sort_values("matrix_size").groupby("algorithm"):
    group = group.sort_values("matrix_size")
    previous = None
    for _, row in group.iterrows():
        if previous is not None:
            ratio = row["time_seconds_mean"] / previous["time_seconds_mean"]
            growth_rows.append({
                "algorithm": algorithm,
                "algorithm_label": algorithm_labels[algorithm],
                "from_n": int(previous["matrix_size"]),
                "to_n": int(row["matrix_size"]),
                "time_ratio": ratio,
                "empirical_exponent": np.log2(ratio),
            })
        previous = row

growth = pd.DataFrame(growth_rows)
display(growth)

fig, ax = plt.subplots(figsize=(9.5, 5.4))
sns.lineplot(
    data=growth,
    x="to_n",
    y="empirical_exponent",
    hue="algorithm_label",
    marker="o",
    hue_order=[algorithm_labels[a] for a in algorithms],
    ax=ax,
)
ax.axhline(3.0, color="#555555", linestyle="--", linewidth=1, label="O(n^3)")
ax.axhline(np.log2(7), color="#111111", linestyle=":", linewidth=1.2, label="O(n^log2 7)")
ax.set_xscale("log", base=2)
ax.set_xlabel("Upper matrix size in transition")
ax.set_ylabel("Empirical exponent")
ax.set_title("Observed Growth Exponent Between Consecutive Sizes")
ax.legend(title="Algorithm", bbox_to_anchor=(1.02, 1), loc="upper left")
save_current_figure("empirical_growth_exponent.png")

## 7. Tracked Heap Usage

The tracked heap metric captures dynamic allocations routed through `tracked_malloc`. This is the right metric for comparing recursive temporary matrix pressure. Values of zero mean the implementation did not allocate through the tracker for that size, which is expected for the iterative baseline and for hybrid algorithms below their threshold.

In [ ]:
heap_positive = summary[summary["heap_peak_bytes_mean"] > 0].copy()
heap_positive["heap_peak_mib_mean"] = heap_positive["heap_peak_bytes_mean"] / (1024 * 1024)

fig, ax = plt.subplots(figsize=(9.5, 5.6))
sns.lineplot(
    data=heap_positive,
    x="matrix_size",
    y="heap_peak_mib_mean",
    hue="algorithm_label",
    marker="o",
    hue_order=[algorithm_labels[a] for a in algorithms],
    ax=ax,
)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Matrix size n")
ax.set_ylabel("Mean tracked heap peak (MiB, log scale)")
ax.set_title("Tracked Heap Peak by Algorithm")
ax.legend(title="Algorithm", bbox_to_anchor=(1.02, 1), loc="upper left")
save_current_figure("heap_peak_mib_loglog.png")

display(summary[["algorithm_label", "matrix_size", "heap_peak_bytes_mean", "heap_allocations_mean"]].sort_values(["matrix_size", "algorithm_label"]))

### Interpretation

Strassen variants allocate more temporary storage than divide-and-conquer variants. This is expected from the algorithm structure: Strassen reduces recursive multiplications but pays for extra matrix additions/subtractions and temporary products.

The iterative baseline is not visible in the positive heap plot because it does not use `tracked_malloc`. That does not mean it uses no memory at all; it means the benchmark is tracking algorithm-created temporary heap allocations, not stack frames or the input/output matrices allocated by the driver.

## 8. Numerical Error

All algorithms were validated against the iterative result for the same generated matrix pair. Exact bitwise equality is not required because floating-point addition is not associative, and Strassen changes the order and structure of operations.

In [ ]:
error_data = summary[summary["max_abs_error_mean"] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.lineplot(
    data=error_data,
    x="matrix_size",
    y="max_abs_error_mean",
    hue="algorithm_label",
    marker="o",
    hue_order=[algorithm_labels[a] for a in algorithms],
    ax=axes[0],
)
axes[0].set_xscale("log", base=2)
axes[0].set_yscale("log")
axes[0].set_title("Mean Maximum Absolute Error")
axes[0].set_xlabel("Matrix size n")
axes[0].set_ylabel("Absolute error")
axes[0].legend_.remove()

sns.lineplot(
    data=error_data,
    x="matrix_size",
    y="max_rel_error_mean",
    hue="algorithm_label",
    marker="o",
    hue_order=[algorithm_labels[a] for a in algorithms],
    ax=axes[1],
)
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")
axes[1].set_title("Mean Maximum Relative Error")
axes[1].set_xlabel("Matrix size n")
axes[1].set_ylabel("Relative error")
axes[1].legend(title="Algorithm", bbox_to_anchor=(1.02, 1), loc="upper left")
save_current_figure("numerical_error_loglog.png")

## 9. Practical Winners

The table below summarizes the fastest algorithm by matrix size and the lowest positive tracked heap peak among algorithms that allocate temporary matrices through the tracker.

In [ ]:
fastest = summary.loc[summary.groupby("matrix_size")["time_seconds_mean"].idxmin(), [
    "matrix_size", "algorithm_label", "time_seconds_mean", "time_seconds_stddev"
]].rename(columns={"algorithm_label": "fastest_algorithm"})

positive_heap = summary[summary["heap_peak_bytes_mean"] > 0]
lowest_positive_heap = positive_heap.loc[positive_heap.groupby("matrix_size")["heap_peak_bytes_mean"].idxmin(), [
    "matrix_size", "algorithm_label", "heap_peak_bytes_mean"
]].rename(columns={"algorithm_label": "lowest_positive_heap_algorithm"})

winners = fastest.merge(lowest_positive_heap, on="matrix_size", how="left")
display(winners)

## 10. Conclusions

When executed against a complete nine-size dataset, use this section to check whether the results align with the expected algorithmic trade-offs:

- Pure divide-and-conquer has cubic asymptotic complexity and very high practical overhead.
- Pure Strassen reduces the multiplication count and improves over pure divide-and-conquer, but recursion to `n == 1` keeps overhead high.
- Hybridization is the decisive practical improvement.
- `hybrid_strassen` is the strongest algorithm for the larger tested sizes.
- Tracked heap usage confirms the expected memory cost of Strassen-style algorithms.
- The smallest sizes are useful for sanity checks but should not dominate scientific conclusions because their runtimes are short and noisier.

For a report, the strongest evidence comes from the larger transitions now added beyond `1024`, while still comparing time trends together with tracked heap peaks and feasibility limits.